# 🎓 AI Tutor
An AI-powered tutor that converts passive technical content into active learning workflows.
**Run cells 1–10 top-to-bottom once**, then interact via **Cell 11** (the tabbed UI).

## Cell 1 — Setup & Dependencies
Installs packages, imports modules, configures the OpenAI client and data directory.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openai', 'graphviz', '-q'], check=False)

import os, json, uuid, random, io
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import Optional

import graphviz
from openai import OpenAI

# ── API Key ───────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    IS_COLAB = True
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
    IS_COLAB = False

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not set. Add it to Colab Secrets or set the env var.')

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Data directory ────────────────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = Path('/content/drive/MyDrive/ai_tutor/data')
else:
    DATA_DIR = Path('./data')


DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature flags ─────────────────────────────────────────────────────────────
ENABLE_WEB_SUGGESTIONS = False  # Set False to skip the post-ingest web search step
VERBOSE_LOGGING        = True   # True: show full ReAct Think/Act/Observe traces.
                                # False: only show final results (clean UX).


def agent_log(*args, **kwargs):
    """log_fn passed to long-running agents. Suppressed when VERBOSE_LOGGING is False."""
    if VERBOSE_LOGGING:
        print(*args, **kwargs)


print(f'✓ Setup complete | Data dir: {DATA_DIR} | Colab: {IS_COLAB}')


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


✓ Setup complete | Data dir: data | Colab: False


## Cell 2 — Data Models & Persistence (`Store`)
Defines `Content`, `Topic`, and `SessionRecord` dataclasses plus the `Store` class that owns all JSON I/O. Every other class reads and writes through `Store`.

In [2]:
@dataclass
class Content:
    id: str
    title: str
    body: str
    topic_id: str
    source_file: str
    created_at: str
    vector_file_id: str = ''


@dataclass
class Topic:
    id: str
    name: str
    parent_id: Optional[str]
    children_topic_ids: list = field(default_factory=list)
    content_ids: list = field(default_factory=list)


@dataclass
class SessionRecord:
    date: str
    content_id: str
    question_type: str   # 'mcq' | 'open'
    correct: bool
    score: float


class Store:
    """Single source of truth for all JSON persistence."""

    TREE_FILE     = 'content_tree.json'
    VS_FILE       = 'vector_store.json'
    PROGRESS_FILE = 'user_progress.json'

    def __init__(self, data_dir: Path):
        self.data_dir = data_dir
        self.topics: dict = {}
        self.contents: dict = {}
        self.vector_store_id: str = ''
        self.sessions: list = []
        self.weak_content_ids: list = []
        self.load()

    # ── I/O ──────────────────────────────────────────────────────────────────

    def load(self):
        self._load_tree()
        self._load_vs()
        self._load_progress()

    def save(self):
        self._save_tree()
        self._save_vs()
        self._save_progress()

    def _load_tree(self):
        path = self.data_dir / self.TREE_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.topics   = {k: Topic(**v)   for k, v in data.get('topics', {}).items()}
            self.contents = {k: Content(**v) for k, v in data.get('contents', {}).items()}
        else:
            root = Topic(id=str(uuid.uuid4()), name='Root', parent_id=None)
            self.topics[root.id] = root
            self._save_tree()

    def _save_tree(self):
        (self.data_dir / self.TREE_FILE).write_text(json.dumps(
            {'topics': {k: asdict(v) for k, v in self.topics.items()},
             'contents': {k: asdict(v) for k, v in self.contents.items()}},
            indent=2
        ))

    def _load_vs(self):
        path = self.data_dir / self.VS_FILE
        if path.exists():
            self.vector_store_id = json.loads(path.read_text()).get('vector_store_id', '')

    def _save_vs(self):
        (self.data_dir / self.VS_FILE).write_text(
            json.dumps({'vector_store_id': self.vector_store_id}, indent=2)
        )

    def _load_progress(self):
        path = self.data_dir / self.PROGRESS_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.sessions          = [SessionRecord(**s) for s in data.get('sessions', [])]
            self.weak_content_ids  = data.get('weak_content_ids', [])
        else:
            self._save_progress()

    def _save_progress(self):
        (self.data_dir / self.PROGRESS_FILE).write_text(json.dumps(
            {'sessions': [asdict(s) for s in self.sessions],
             'weak_content_ids': self.weak_content_ids},
            indent=2
        ))

    # ── Accessors ─────────────────────────────────────────────────────────────

    def get_topic(self, id: str) -> Optional[Topic]:
        return self.topics.get(id)

    def get_content(self, id: str) -> Optional[Content]:
        return self.contents.get(id)

    def add_topic(self, topic: Topic):
        self.topics[topic.id] = topic
        self._save_tree()

    def add_content(self, content: Content):
        self.contents[content.id] = content
        self._save_tree()

    def delete_content(self, id: str):
        content = self.contents.pop(id, None)
        if content:
            topic = self.topics.get(content.topic_id)
            if topic and id in topic.content_ids:
                topic.content_ids.remove(id)
            self._save_tree()
        return content

    def all_content(self) -> list:
        return list(self.contents.values())

    def all_topics(self) -> list:
        return list(self.topics.values())

    def root_topic(self) -> Optional[Topic]:
        return next((t for t in self.topics.values() if t.parent_id is None), None)


store = Store(DATA_DIR)
print(f'✓ Store loaded | Topics: {len(store.topics)} | Content items: {len(store.contents)}')

✓ Store loaded | Topics: 29 | Content items: 118


## Cell 3 — Content Management (`ContentManager`)
Wraps `Store` to provide topic/content tree CRUD, navigation, and graphviz tree visualisation.
`render_tree()` returns a `graphviz.Digraph`; call `display(cm.render_tree())` to show it inline.

In [3]:
class ContentManager:
    """Manages navigation and editing of the topic/content tree."""

    def __init__(self, store: Store):
        self.store = store

    def create_topic(self, name: str, parent_id: str = None) -> Topic:
        if parent_id is None:
            root = self.store.root_topic()
            parent_id = root.id if root else None
        topic = Topic(id=str(uuid.uuid4()), name=name, parent_id=parent_id)
        self.store.add_topic(topic)
        if parent_id:
            parent = self.store.get_topic(parent_id)
            if parent and topic.id not in parent.children_topic_ids:
                parent.children_topic_ids.append(topic.id)
                self.store._save_tree()
        return topic

    def add_content_to_topic(self, content: Content, topic_id: str):
        self.store.add_content(content)
        topic = self.store.get_topic(topic_id)
        if topic and content.id not in topic.content_ids:
            topic.content_ids.append(content.id)
            self.store._save_tree()

    def list_topics(self, parent_id: str = None, indent: int = 0) -> str:
        if parent_id is None:
            root = self.store.root_topic()
            if not root:
                return '(empty)'
            parent_id = root.id
        topic = self.store.get_topic(parent_id)
        if not topic:
            return ''
        lines = ['  ' * indent + f'📁 {topic.name}']
        for cid in topic.content_ids:
            c = self.store.get_content(cid)
            if c:
                lines.append('  ' * (indent + 1) + f'📄 {c.title}')
        for child_id in topic.children_topic_ids:
            lines.append(self.list_topics(child_id, indent + 1))
        return '\n'.join(lines)

    def view_content(self, content_id: str) -> Optional[Content]:
        return self.store.get_content(content_id)

    def delete_content(self, content_id: str) -> str:
        """Returns vector_file_id for upstream cleanup."""
        content = self.store.delete_content(content_id)
        return content.vector_file_id if content else ''

    def move_content(self, content_id: str, new_topic_id: str):
        content = self.store.get_content(content_id)
        if not content:
            return
        old_topic = self.store.get_topic(content.topic_id)
        if old_topic and content_id in old_topic.content_ids:
            old_topic.content_ids.remove(content_id)
        content.topic_id = new_topic_id
        new_topic = self.store.get_topic(new_topic_id)
        if new_topic and content_id not in new_topic.content_ids:
            new_topic.content_ids.append(content_id)
        self.store._save_tree()


    def update_content(self, content_id: str, new_title: str = None, new_body: str = None) -> 'Content | None':
        """Update title and/or body of an existing content item. Returns updated Content or None."""
        content = self.store.get_content(content_id)
        if not content:
            return None
        if new_title is not None:
            content.title = new_title
        if new_body is not None:
            content.body = new_body
        self.store._save_tree()
        return content
    def render_tree(self):
        """Returns a graphviz.Digraph — call display(cm.render_tree()) to render inline."""
        dot = graphviz.Digraph(graph_attr={'rankdir': 'TB', 'splines': 'ortho'})
        dot.attr('node', fontname='Helvetica')
        for topic in self.store.all_topics():
            dot.node(topic.id, label=f'📁 {topic.name}', shape='folder',
                     style='filled', fillcolor='#dce8f5')
            if topic.parent_id:
                dot.edge(topic.parent_id, topic.id)
            for cid in topic.content_ids:
                content = self.store.get_content(cid)
                if content:
                    dot.node(cid, label=f'📄 {content.title}', shape='note',
                             style='filled', fillcolor='#f5f0dc')
                    dot.edge(topic.id, cid)
        return dot


content_manager = ContentManager(store)
print('✓ ContentManager ready')

✓ ContentManager ready


## Cell 4 — ReAct Agent (`Agent`)
Reusable ReAct engine. Subclass it to build agents that follow the
**Think → Act → Observe** loop using OpenAI function calling.

- `_fn_tool` — builds a strict function-tool definition
- `_think` — one LLM call; returns the single function-call decision
- `_act` — executes a tool call; returns the observation string
- `_react_loop` — orchestrates the loop until the finalization tool is called

In [4]:
class Agent:
    """Reusable ReAct engine: Think → Act → Observe via OpenAI function calling."""

    MAX_REACT_CYCLES = 10

    def __init__(self, openai_client):
        self.client = openai_client

    # ── Tool definition ───────────────────────────────────────────────────────

    @staticmethod
    def _fn_tool(name: str, description: str, properties: dict, required: list = None) -> dict:
        """Build a function tool definition following the OpenAI function-calling spec."""
        return {
            'type': 'function',
            'name': name,
            'description': description,
            'parameters': {
                'type': 'object',
                'properties': properties,
                'required': required or [],
                'additionalProperties': False,
            },
            'strict': True,
        }

    # ── ReAct phases ──────────────────────────────────────────────────────────

    def _think(self, next_input: list, tools: list, prev_id: str | None):
        """
        Think phase: call the model and return its function-call decisions.

        Returns (response_id, fn_calls, had_ci_call).
        fn_calls: list of {name, call_id, args} for custom function tools.
        had_ci_call: True when the model used the built-in Code Interpreter
                     this cycle (executed server-side; no client action needed).
        parallel_tool_calls=False keeps one tool call per cycle.
        """
        kwargs = dict(model='gpt-4o', input=next_input, tools=tools, parallel_tool_calls=False)
        if prev_id:
            kwargs['previous_response_id'] = prev_id
        response = self.client.responses.create(**kwargs)

        response_id = getattr(response, 'id', None)
        fn_calls    = []
        had_ci_call = False
        for item in getattr(response, 'output', []) or []:
            item_type = getattr(item, 'type', None)
            if item_type == 'function_call':
                raw = getattr(item, 'arguments', '{}') or '{}'
                try:
                    args = json.loads(raw)
                except Exception:
                    args = {}
                fn_calls.append({
                    'name':    getattr(item, 'name', ''),
                    'call_id': getattr(item, 'call_id', ''),
                    'args':    args,
                })
            elif item_type == 'code_interpreter_call':
                had_ci_call = True
        return response_id, fn_calls, had_ci_call

    def _act(self, call: dict, execute_fn) -> str:
        """Act phase: execute one tool call and return the observation string."""
        return execute_fn(call['name'], call['args'])

    # ── Loop orchestrator ─────────────────────────────────────────────────────

    def _react_loop(
        self,
        initial_input: list,
        tools: list,
        final_tool_name: str,
        execute_fn,
        step_name: str,
        log_fn=None,
        show_reasoning: bool = True,
    ) -> dict:
        """
        Orchestrate Think → Act → Observe until `final_tool_name` is called.

        Each cycle: _think() returns one decision; if it is a regular tool,
        _act() executes it and the observation feeds the next cycle; if it is
        `final_tool_name`, its arguments are returned as the step result.
        Conversation state persists across cycles via previous_response_id.

        Args:
            initial_input:    First messages sent to the model.
            tools:            Tool definitions available this step.
            final_tool_name:  Calling this tool ends the loop.
            execute_fn:       fn(tool_name, args) -> observation str.
            step_name:        Label used in log lines.
            log_fn:           Callable for log output; defaults to print.
            show_reasoning:   Whether to emit [Think] lines.
        """
        _log       = log_fn or print
        prev_id    = None
        next_input = initial_input

        for cycle in range(1, self.MAX_REACT_CYCLES + 1):
            try:
                prev_id, calls, had_ci_call = self._think(next_input, tools, prev_id)
            except Exception as exc:
                _log(f'    [Think   {cycle}] ✗ API error: {exc}')
                break

            if not calls:
                if had_ci_call:
                    _log(f'    [Observe {cycle}] 🖥️  Code Interpreter executed (server-side).')
                    next_input = []
                else:
                    _log(f'    [Think   {cycle}] No tool selected — prompting continuation.')
                    next_input = [{
                        'role': 'user',
                        'content': (
                            f'No tool was called. Continue reasoning, use a tool, '
                            f'or call `{final_tool_name}` when satisfied.'
                        ),
                    }]
                continue

            tool_outputs = []
            final_result = None

            for call in calls:
                name    = call['name']
                call_id = call['call_id']
                args    = call['args']
                display = {k: v for k, v in args.items() if k != 'thought'}

                thought = (args.get('thought') or '').strip()
                if thought and show_reasoning:
                    _log(f'    [Think   {cycle}] 💭 {thought}')

                _log(f'    [Act     {cycle}] ⚙️  {name}({json.dumps(display, ensure_ascii=True)})')

                if name == final_tool_name:
                    final_result = args
                    if call_id:
                        tool_outputs.append({
                            'type':    'function_call_output',
                            'call_id': call_id,
                            'output':  'Submission accepted.',
                        })
                else:
                    observation = self._act(call, execute_fn)
                    _log(f'    [Observe {cycle}] 👀 {str(observation)}')
                    if call_id:
                        tool_outputs.append({
                            'type':    'function_call_output',
                            'call_id': call_id,
                            'output':  str(observation),
                        })

            if final_result is not None:
                _log(f'    [Done] {step_name} complete at cycle {cycle}')
                return final_result

            next_input = tool_outputs

        _log(f'    [Done] {step_name}: max cycles reached without completion.')
        return {}


print('✓ Agent class defined')


✓ Agent class defined


## Cell 5 — Shared Tool Definitions (`ToolDefs`)

In [5]:
class ToolDefs:
    """Canonical tool definitions shared across agents. Reference as ToolDefs.LIST_TREE etc."""

    LIST_TREE = Agent._fn_tool(
        'list_tree',
        'Return the full topic/content tree as an indented text outline.',
        {'thought': {'type': 'string', 'description': 'Reasoning before inspecting.'}},
        ['thought'],
    )

    SEARCH_CONTENT = Agent._fn_tool(
        'search_content',
        'Semantic search across ingested content. Returns top_k matches as JSON [{id, title, snippet}].',
        {
            'thought': {'type': 'string'},
            'query':   {'type': 'string', 'description': 'Search query.'},
            'top_k':   {'type': 'integer', 'description': 'Maximum number of results to return.'},
        },
        required=['thought', 'query', 'top_k'],
    )

    CREATE_TOPIC = Agent._fn_tool(
        'create_topic',
        'Create a new topic. Call list_tree first to confirm the right parent.',
        {
            'thought':    {'type': 'string', 'description': 'Reasoning before creating.'},
            'name':       {'type': 'string', 'description': 'Name for the new topic.'},
            'parent_ref': {'type': 'string', 'description': 'ID or name of parent topic (empty = root).'},
        },
        required=['thought', 'name', 'parent_ref'],
    )


print('✓ ToolDefs defined')

✓ ToolDefs defined


## Cell 6 — Content Ingestion (`Ingester`)

In [6]:
class Ingester(Agent):
    """ReAct agent for content ingestion: upload → outline → categorize → store → embed."""

    SUPPORTED_EXTS = {'.pdf', '.png', '.jpg', '.jpeg', '.webp', '.gif'}
    def __init__(self, openai_client, content_manager, vector_store, web_searcher,
                 suggest_similar: bool = True, show_reasoning: bool = True):
        super().__init__(openai_client)
        self.cm              = content_manager
        self.vs              = vector_store
        self.ws              = web_searcher
        self.suggest_similar = suggest_similar
        self.show_reasoning  = show_reasoning
        self._active_log_fn  = print

    # ── Public entry point ────────────────────────────────────────────────────

    def ingest(self, file_bytes: bytes, filename: str, log_fn=None) -> list:
        """Full pipeline: upload → ReAct outline → ReAct categorize → store → embed → suggest → cleanup."""
        ext = Path(filename).suffix.lower()
        if ext not in self.SUPPORTED_EXTS:
            raise ValueError(f'Unsupported type "{ext}". Supported: {self.SUPPORTED_EXTS}')

        prev = self._active_log_fn
        self._active_log_fn = log_fn or print
        try:
            self._log(f'⬆  Uploading "{filename}"…')
            file_id = self._upload_file(file_bytes, filename)

            self._log('🧠 ReAct — Step 1/2: Draft outline…')
            _topics_before = len(self.cm.store.topics)
            outline_topics = self._run_outline_step(file_id)
            new_topics = len(self.cm.store.topics) - _topics_before
            self._log(f'  ✓ {new_topics} topic(s) created, {len(outline_topics)} in outline')

            self._log('🧠 ReAct — Step 2/2: Categorize content…')
            self._current_filename = filename
            created = self._run_categorization_step(
                file_id=file_id,
                outline_topics=outline_topics,
            )
            self._log(f'  ✓ {len(created)} concept(s) across {len(outline_topics)} topic(s)')

            if self.suggest_similar:
                self._log('🔍 Searching for related learning resources…')
                for c in created:
                    suggestions = self.ws.suggest(c.title)
                    if suggestions:
                        self._log(f'\n  📌 Resources for "{c.title}":')
                        for s in suggestions[:3]:
                            self._log(f'     • {s.get("title", "")}')
                            self._log(f'       {s.get("url", "")}')

            self._log('\n🗑  Cleaning up upload…')
            self._delete_upload(file_id)
            self._log(
                f'\n✅ Ingestion complete — '
                f'{new_topics} topic(s) created, '
                f'{self._cat_new_count} concept(s) added, '
                f'{self._cat_merged_count} merged into existing.'
            )
            return created
        finally:
            self._active_log_fn = prev

    def _outline_tools(self) -> list:
        return [
                 ToolDefs.LIST_TREE,
                 ToolDefs.CREATE_TOPIC,
            self._fn_tool(
                'submit_outline',
                (
                    'Submit the final list of topics for this document. '
                    'Each entry must have a real topic_id that already exists in the tree '
                    '(either reused or just created with create_topic). '
                    'Terminates the ReAct loop for this step.'
                ),
                {
                    'thought': {
                        'type': 'string',
                        'description': 'Reasoning for the topic structure chosen.',
                    },
                    'topics': {
                        'type': 'array',
                        'description': 'Topics covering this document — both reused and newly created.',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'topic_id':   {'type': 'string', 'description': 'Real ID in the content tree.'},
                                'topic_name': {'type': 'string', 'description': 'Display name.'},
                            },
                            'required': ['topic_id', 'topic_name'],
                            'additionalProperties': False,
                        },
                    },
                },
                ['thought', 'topics'],
            ),
        ]

    def _categorize_tools(self, outline_topics: list) -> list:
        topic_ids_str = ', '.join(
            f'{t["topic_name"]} (id={t["topic_id"]})'
            for t in outline_topics
        )
        return [
            ToolDefs.SEARCH_CONTENT,
            self._fn_tool(
                'show_existing_content',
                'Retrieve the full title and body of an existing content item by its ID.',
                {
                    'thought':    {'type': 'string', 'description': 'Why you are inspecting this item.'},
                    'content_id': {'type': 'string', 'description': 'ID of the content item to inspect.'},
                },
                required=['thought', 'content_id'],
            ),
            self._fn_tool(
                'create_content',
                'Store a brand-new concept in the tree and index it in the vector store immediately.',
                {
                    'thought':  {'type': 'string', 'description': 'Why no existing item is close enough to merge.'},
                    'title':    {'type': 'string', 'description': 'Short concept title.'},
                    'body':     {'type': 'string', 'description': 'Detailed body suitable for question generation.'},
                    'topic_id': {'type': 'string', 'description': f'Must be one of: {topic_ids_str}.'},
                },
                required=['thought', 'title', 'body', 'topic_id'],
            ),
            self._fn_tool(
                'update_content',
                'Merge new material into an existing concept: update its title/body and re-index immediately.',
                {
                    'thought':    {'type': 'string', 'description': 'What is being merged and why.'},
                    'content_id': {'type': 'string', 'description': 'ID of the existing content item to update.'},
                    'new_title':  {'type': 'string', 'description': 'Updated title; pass the existing title to leave it unchanged.'},
                    'new_body':   {'type': 'string', 'description': 'Merged body text.'},
                },
                required=['thought', 'content_id', 'new_title', 'new_body'],
            ),
            self._fn_tool(
                'submit_categorization',
                'Signal that every concept in the document has been processed. Ends the loop.',
                {
                    'thought': {'type': 'string', 'description': 'Brief summary of what was created/merged.'},
                },
                required=['thought'],
            ),
        ]
    # ── Step 1: Draft outline ─────────────────────────────────────────────────

    def _run_outline_step(self, file_id: str) -> list:
        """
        Outline step: agent reads the document, inspects the tree, creates any needed
        topics via add_topic, then submits a concrete list of {topic_id, topic_name}.
        Returns that list — all IDs are already live in the content tree.
        """
        tools = self._outline_tools()
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a ReAct ingestion agent.\n'
                        'Goal: place this document into a well-grouped, nested topic hierarchy and '
                        'return the topics that concepts will be directly assigned to.\n\n'
                        'Tree structure rules:\n'
                        '  - Root-level topics represent broad Computer Science domains '
                        '    (e.g. Programming Languages, Databases, Algorithms & Data Structures, '
                        '    Operating Systems, Networks, Machine Learning, Software Engineering).\n'
                        '  - Group related topics under a shared parent whenever possible — '
                        '    prefer nesting over a flat list of siblings under Root.\n'
                        '  - Depth is flexible: two levels may be enough for a broad topic, '
                        '    three or more for a specific one. Let the content dictate the depth.\n'
                        '  - Topics submitted in submit_outline are the ones concepts attach to; '
                        '    pure grouping parents do not need to be included.\n\n'
                        'How to work:\n'
                        '  - One tool call at a time. Always read the observation before deciding the next action.\n'
                        '  - Use the `thought` field to reason about what the observation tells you \n'
                        '    and what you will do next — especially when a tool returns an error.\n'
                        '  - If a tool call fails, diagnose the error from the observation and fix it \n'
                        '    (e.g. call `list_tree` to get valid IDs) before retrying.\n'
                        '  - Start with `list_tree`, then create missing topics top-down \n'
                        '    (broader groups first, using the real ID returned by each `add_topic`).\n'
                        '  - Call `submit_outline` only when all needed topics exist in the tree.\n\n'
                        'Quality criteria:\n'
                        '  - Every topic_id in submit_outline already exists in the tree.\n'
                        '  - Root children are broad CS domains, not document-specific titles.\n'
                        '  - No near-duplicate topics at any level.\n'
                        '  - Topics are specific enough for concept-level categorization.\n'
                    ),
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'input_file', 'file_id': file_id},
                        {'type': 'input_text', 'text': (
                            'Read this document and build its topic hierarchy.\n'
                            'Think before every action (use the `thought` field), and always \n'
                            'react to what the observation tells you before calling the next tool.'
                        )},
                    ],
                },
            ],
            tools=tools,
            final_tool_name='submit_outline',
            execute_fn=lambda name, args: self._execute_tool(name, args),
            step_name='outline',
            log_fn=self._log,
            show_reasoning=self.show_reasoning,
        )
        if self.show_reasoning and result.get('thought'):
            self._log(f'    💭 Final thought: {result["thought"][:200]}')
        return result.get('topics', [])

    # ── Step 2: Categorize content ────────────────────────────────────────────

    def _run_categorization_step(self, file_id: str, outline_topics: list) -> list:
        """
        Categorize step: agent assigns every concept in the document to one of the
        topics produced by the outline step. No new topics are created here.
        Returns a flat list of concept dicts with {title, body, topic_id}.
        """
        self._cat_created: list = []
        self._cat_new_count:     int  = 0
        self._cat_merged_count:  int  = 0
        tools = self._categorize_tools(outline_topics)
        _prev_max, self.MAX_REACT_CYCLES = self.MAX_REACT_CYCLES, 50
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a ReAct ingestion agent.\n'
                        'Goal: extract every distinct concept from the document and immediately '
                        'store or merge each one as you go.\n\n'
                        'MANDATORY per-concept workflow — repeat for every concept:\n'
                        '  1. Identify the next concept from the document.\n'
                        '  2. Call `search_content` with the concept title as the query (top_k=3).\n'
                        '  3. If results are returned, call `show_existing_content` on the closest match\n'
                        '     to read its full body.\n'
                        '  4. Decide and ACT immediately:\n'
                        '     - If the existing item covers the same concept → call `update_content` to\n'
                        '       merge the new material in (write a unified body combining both).\n'
                        '     - Otherwise → call `create_content` with the full title and body.\n'
                        '  5. Move on to the next concept.\n\n'
                        'After ALL concepts have been processed, call `submit_categorization` to end.\n\n'
                        'Rules:\n'
                        '  - Never batch — act on each concept before moving to the next.\n'
                        '  - Use only the topic_ids from the outline for `create_content`.\n'
                        '  - Concept bodies must be detailed enough for question generation.\n'
                        '  - Use `thought` to reason explicitly about each merge/create decision.\n'
                    ),
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'input_file', 'file_id': file_id},
                        {'type': 'input_text', 'text': (
                            f'Outline topics (use only these IDs):\n'
                            f'{json.dumps(outline_topics, ensure_ascii=True)}\n\n'
                            'Extract all concepts from the document and assign each to one of the '
                            'topic_ids above. Call `submit_categorization` when done.'
                        )},
                    ],
                },
            ],
            tools=tools,
            final_tool_name='submit_categorization',
            execute_fn=lambda name, args: self._execute_tool(name, args),
            step_name='categorize',
            log_fn=self._log,
            show_reasoning=self.show_reasoning,
        )
        self.MAX_REACT_CYCLES = _prev_max
        if self.show_reasoning and result.get('thought'):
            self._log(f'    💭 Final thought: {result["thought"][:200]}')
        self.MAX_REACT_CYCLES = _prev_max
        return self._cat_created

    # ── Tool executor ─────────────────────────────────────────────────────────

    def _execute_tool(self, tool_name: str, args: dict) -> str:
        """Dispatch a tool call and return an observation string."""
        tool_name = (tool_name or '').strip().lower()
        args      = args or {}

        if tool_name == 'list_tree':
            snapshot = self._topic_state()
            return f'{len(snapshot)} topic(s) in tree: {json.dumps(snapshot, ensure_ascii=True)}'

        if tool_name == 'create_topic':
            topic_name = (args.get('name') or '').strip()
            parent_id  = (args.get('parent_ref') or '').strip()
            if not topic_name:
                return 'Error: name is required.'
            if not parent_id or not self.cm.store.get_topic(parent_id):
                return (
                    'Error: parent_ref is missing or invalid. '
                    'Call list_tree to find valid topic IDs.'
                )
            existing = self._find_topic_by_name(topic_name)
            if existing:
                return (
                    f'Skipped: topic "{existing.name}" ({existing.id}) already exists — '
                    f'reuse it instead of creating a duplicate.'
                )
            created = self.cm.create_topic(topic_name, parent_id)
            return f'Created: "{created.name}" (id={created.id}) under parent={parent_id}.'

        if tool_name == 'search_content':
            query = (args.get('query') or '').strip()
            top_k = int(args.get('top_k') or 5)
            if not query:
                return 'Error: query is required.'
            if not self.vs.store.all_content():
                return 'No existing content to search.'
            ids = self.vs.search(query, top_k=top_k)
            if not ids:
                return 'No matching content found.'
            rows = []
            for cid in ids:
                c = self.cm.store.get_content(cid)
                if c:
                    topic = self.cm.store.get_topic(c.topic_id)
                    rows.append({'id': c.id, 'title': c.title,
                                 'topic': topic.name if topic else '?'})
            return json.dumps(rows)

        if tool_name == 'show_existing_content':
            cid = (args.get('content_id') or '').strip()
            if not cid:
                return 'Error: content_id is required.'
            c = self.cm.store.get_content(cid)
            if not c:
                return f'Error: content item {cid!r} not found.'
            return json.dumps({'id': c.id, 'title': c.title, 'body': c.body})

        if tool_name == 'create_content':
            title_val = (args.get('title') or '').strip()
            body_val  = (args.get('body') or '').strip()
            topic_id  = (args.get('topic_id') or '').strip()
            if not title_val:
                return 'Error: title is required.'
            if not body_val:
                return 'Error: body is required.'
            if not topic_id or not self.cm.store.get_topic(topic_id):
                return f'Error: topic_id {topic_id!r} not found in tree.'
            c = Content(
                id=str(uuid.uuid4()),
                title=title_val,
                body=body_val,
                topic_id=topic_id,
                source_file=self._current_filename,
                created_at=datetime.utcnow().isoformat(),
            )
            c.vector_file_id = self.vs.add_content(c)
            self.cm.add_content_to_topic(c, topic_id)
            self._cat_created.append(c)
            self._cat_new_count += 1
            topic = self.cm.store.get_topic(topic_id)
            self._log(f'  ✚ Created: "{c.title}" → {topic.name if topic else topic_id}')
            return f'Created "{c.title}" (id={c.id}) in topic {topic_id}.'

        if tool_name == 'update_content':
            content_id = (args.get('content_id') or '').strip()
            new_title  = (args.get('new_title') or '').strip() or None
            new_body   = (args.get('new_body') or '').strip() or None
            if not content_id:
                return 'Error: content_id is required.'
            if not new_body:
                return 'Error: new_body is required.'
            existing = self.cm.store.get_content(content_id)
            if not existing:
                return f'Error: content {content_id!r} not found.'
            old_file_id = existing.vector_file_id
            updated = self.cm.update_content(content_id, new_title=new_title, new_body=new_body)
            if not updated:
                return f'Error: failed to update {content_id!r}.'
            updated.vector_file_id = self.vs.update_content(updated, old_file_id)
            self.cm.store._save_tree()
            self._cat_created.append(updated)
            self._cat_merged_count += 1
            self._log(f'  🔀 Merged into: "{updated.title}"')
            return f'Updated "{updated.title}" (id={updated.id}); re-indexed in vector store.'

        return f'Unknown tool: {tool_name}'


    # ── Persistence helpers ───────────────────────────────────────────────────

    def _topic_state(self) -> list:
        return [
            {'id': t.id, 'name': t.name, 'parent_id': t.parent_id}
            for t in self.cm.store.all_topics()
        ]

    def _find_topic_by_name(self, name: str):
        key = self._normalize(name)
        for t in self.cm.store.all_topics():
            if self._normalize(t.name) == key:
                return t
        return None

    def _find_topic_by_name_under_parent(self, name: str, parent_id: str):
        key = self._normalize(name)
        for t in self.cm.store.all_topics():
            if t.parent_id == parent_id and self._normalize(t.name) == key:
                return t
        return None

    @staticmethod
    def _normalize(text: str) -> str:
        return (text or '').strip().lower()

    # ── File API helpers ──────────────────────────────────────────────────────

    def _upload_file(self, file_bytes: bytes, filename: str) -> str:
        ext  = Path(filename).suffix.lower()
        mime = 'application/pdf' if ext == '.pdf' else 'image/png'
        resp = self.client.files.create(file=(filename, file_bytes, mime), purpose='assistants')
        return resp.id

    def _delete_upload(self, file_id: str):
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass

    # ── Logging ───────────────────────────────────────────────────────────────

    def _log(self, message: str):
        try:
            self._active_log_fn(message)
        except Exception:
            print(message)

    @staticmethod
    def _parse_json(text: str) -> dict:
        """Strip markdown fences and parse the first JSON object/array found."""
        import re
        text = text.strip()
        # Remove ```json ... ``` or ``` ... ``` fences
        text = re.sub(r'^```[a-zA-Z]*\n?', '', text)
        text = re.sub(r'```$', '', text.strip())
        # Find the outermost { } or [ ]
        for start_ch, end_ch in [('{', '}'), ('[', ']')]:
            s = text.find(start_ch)
            if s == -1:
                continue
            depth = 0
            for e, ch in enumerate(text[s:], start=s):
                if ch == start_ch:
                    depth += 1
                elif ch == end_ch:
                    depth -= 1
                    if depth == 0:
                        return json.loads(text[s:e+1])
        return json.loads(text)


print('✓ Ingester defined (ReAct agent with OpenAI function calling)')


✓ Ingester defined (ReAct agent with OpenAI function calling)


## Cell 7 — Vector Store (`VectorStore`)

In [7]:
class VectorStore:
    """Wraps the OpenAI Vector Stores API for semantic search over ingested content."""

    STORE_NAME = 'ai_tutor'

    def __init__(self, openai_client, store: Store):
        self.client = openai_client
        self.store  = store

    def get_or_create(self) -> str:
        """Return vector store ID, creating one on first call."""
        if self.store.vector_store_id:
            return self.store.vector_store_id
        vs = self.client.vector_stores.create(name=self.STORE_NAME)
        self.store.vector_store_id = vs.id
        self.store._save_vs()
        print(f'✓ Vector store created: {vs.id}')
        return vs.id

    def add_content(self, content: Content) -> str:
        """
        Upload concept body as plain text and attach to vector store. Returns file_id.
        Polls until indexing completes so the concept is searchable immediately.
        """
        vs_id    = self.get_or_create()
        upload_name = f'{content.id}__{content.title}.txt'.replace('/', '_')
        file_obj = self.client.files.create(
            file=(upload_name, content.body.encode('utf-8'), 'text/plain'),
            purpose='assistants'
        )
        try:
            self.client.vector_stores.files.create_and_poll(
                vector_store_id=vs_id, file_id=file_obj.id
            )
        except Exception as exc:
            print(f'  ⚠  vector_stores.files.create_and_poll failed: {exc}')
            try:
                self.client.vector_stores.files.create(
                    vector_store_id=vs_id, file_id=file_obj.id
                )
            except Exception:
                pass
        return file_obj.id

    def delete_content(self, file_id: str):
        """Remove from vector store and Files API (best-effort)."""
        if not file_id:
            return
        vs_id = self.store.vector_store_id
        if vs_id:
            try:
                self.client.vector_stores.files.delete(vector_store_id=vs_id, file_id=file_id)
            except Exception:
                pass
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass


    def update_content(self, content: 'Content', old_file_id: str) -> str:
        """Replace vector-store entry with updated content body. Returns new file_id."""
        self.delete_content(old_file_id)
        return self.add_content(content)
    def search(self, query: str, top_k: int = 5) -> list:
        """
        Return content_ids of the top-k semantically matching concepts.
        Uses the dedicated vector_stores.search endpoint and maps file_id back
        to content_id via Content.vector_file_id — robust to title rewording.
        """
        vs_id = self.get_or_create()
        if not self.store.all_content():
            return []

        file_to_content = {
            c.vector_file_id: c.id
            for c in self.store.all_content()
            if c.vector_file_id
        }
        if not file_to_content:
            return []

        try:
            resp = self.client.vector_stores.search(
                vector_store_id=vs_id,
                query=query,
                max_num_results=min(max(top_k * 2, top_k), 50),
                rewrite_query=True,
            )
        except Exception as exc:
            print(f'  ⚠  vector_stores.search failed: {exc}')
            return []

        seen = set()
        ids  = []
        for r in (getattr(resp, 'data', None) or []):
            fid = getattr(r, 'file_id', None)
            cid = file_to_content.get(fid)
            if cid and cid not in seen:
                ids.append(cid)
                seen.add(cid)
            if len(ids) >= top_k:
                break
        return ids


vector_store = VectorStore(client, store)
print('✓ VectorStore ready')

✓ VectorStore ready


## Cell 8 — Web Search (`WebSearcher`)

In [8]:
class WebSearcher:
    """Surfaces related resources via OpenAI Responses API web_search_preview."""

    def __init__(self, openai_client):
        self.client = openai_client

    def suggest(self, concept_title: str) -> list:
        """Return up to 5 relevant resources for the given concept title."""
        try:
            response = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'web_search_preview'}],
                input=(
                    f'Find the best 3-5 online learning resources, tutorials, or official docs '
                    f'for someone studying: "{concept_title}".\n'
                    'Return ONLY a JSON array — no markdown, no commentary:\n'
                    '[{"title": "...", "url": "...", "summary": "..."}]'
                )
            )
            return Ingester._parse_json(response.output_text)
        except Exception:
            return []


web_searcher = WebSearcher(client)
print('✓ WebSearcher ready')

✓ WebSearcher ready


## Cell 9 — Study Preparation Subagents (`ContentRetriever`, `QuestionEngine`)

In [9]:
class ContentRetriever(Agent):
    """ReAct subagent that picks concepts from the vector store for a study request."""

    MAX_FIND_RESULTS = 5

    def __init__(self, openai_client, store, vector_store, content_manager):
        super().__init__(openai_client)
        self.store           = store
        self.vector_store    = vector_store
        self.content_manager = content_manager

    def find_relevant_content(self, query: str, log_fn=None) -> list:
        """
        Take a natural-language study request and return content_ids of the most
        relevant concepts retrieved from the vector store, ordered most → least
        relevant. Returns an empty list when nothing matches.
        """
        _log = log_fn or print
        _log(f'\n🧠 ReAct — Finding relevant content for: {query!r}')

        if not self.store.all_content():
            _log('  ⚠  No content available — ingest documents first.')
            return []

        def execute_fn(name, args):
            if name == 'list_tree':
                tree = self.content_manager.list_topics()
                return tree or '(empty tree)'
            if name == 'search_content':
                q = (args.get('query') or '').strip()
                if not q:
                    return 'Empty query — provide a search phrase.'
                top_k = int(args.get('top_k') or 8)
                content_ids = self.vector_store.search(q, top_k=top_k)
                if not content_ids:
                    return 'No matches found.'
                results = []
                for cid in content_ids:
                    c = self.store.get_content(cid)
                    if not c:
                        continue
                    snippet = ' '.join(c.body.split())
                    if len(snippet) > 240:
                        snippet = snippet[:240] + '…'
                    results.append({'id': c.id, 'title': c.title, 'snippet': snippet})
                return json.dumps(results, ensure_ascii=False)
            return f'Unknown tool: {name}'

        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a study-content selection agent.\n'
                        'Goal: from the student\'s learning request, pick the concepts in\n'
                        'the vector store that best match what they want to study.\n\n'
                        'Tools available:\n'
                        '  - list_tree             — inspect the topic/content tree to see\n'
                        '                            what subjects exist in the library.\n'
                        '                            Use this FIRST when the request is short,\n'
                        '                            ambiguous, or domain-uncertain (e.g. "classes",\n'
                        '                            "trees", "joins") so you can read the\n'
                        '                            student\'s likely context from the tree.\n'
                        '  - search_content        — semantic search by phrase.\n'
                        '  - submit_content_selection — final answer.\n\n'
                        'Workflow:\n'
                        '  1. (Optional) Call list_tree to ground yourself in what the\n'
                        '     library actually covers and disambiguate the request.\n'
                        '  2. Call search_content with phrasings of the request. Multiple\n'
                        '     targeted searches are fine when the request covers several\n'
                        '     angles or the first hits look weak.\n'
                        '  3. Read returned title + snippet to judge relevance.\n'
                        '  4. Call submit_content_selection with content_ids ordered\n'
                        f'     most → least relevant (up to {self.MAX_FIND_RESULTS}). Pick fewer\n'
                        '     if only a few clearly fit. Submit an empty list if nothing fits.\n\n'
                        'Rules:\n'
                        '  - Only return content_ids that came from search_content results.\n'
                        '  - Do not invent or guess content_ids.\n'
                        '  - Prefer precision over recall — skip weak matches rather than pad.'
                    ),
                },
                {
                    'role': 'user',
                    'content': f'Student wants to learn about: {query}',
                },
            ],
            tools=[
                     ToolDefs.LIST_TREE,
                     ToolDefs.SEARCH_CONTENT,
                self._fn_tool(
                    'submit_content_selection',
                    'Finalise the selection of concepts to study, ordered most → least relevant.',
                    {
                        'thought':     {'type': 'string', 'description': 'Why these concepts together address the request.'},
                        'content_ids': {
                            'type': 'array',
                            'items': {'type': 'string'},
                            'description': f'content_ids from earlier search_content results, in priority order. Up to {self.MAX_FIND_RESULTS}.',
                        },
                    },
                    ['thought', 'content_ids'],
                ),
            ],
            final_tool_name='submit_content_selection',
            execute_fn=execute_fn,
            step_name='find_content',
            log_fn=log_fn,
            show_reasoning=True,
        )

        raw_ids = result.get('content_ids') or []
        seen    = set()
        ids     = []
        for cid in raw_ids:
            if cid in seen:
                continue
            if self.store.get_content(cid):
                ids.append(cid)
                seen.add(cid)
        ids = ids[:self.MAX_FIND_RESULTS]

        if ids:
            _log(f'  ✓ {len(ids)} concept(s) selected:')
            for i, cid in enumerate(ids, 1):
                c = self.store.get_content(cid)
                if c:
                    _log(f'     {i}. {c.title}')
        else:
            _log('  ⚠  No matching concepts.')
        return ids


class QuestionEngine(Agent):
    """ReAct subagent that designs a personalised question set for a study concept."""

    MAX_FOLLOWUPS = 3

    # ── Public API ─────────────────────────────────────────────────────────────

    def design_question_set(self, contents: list, log_fn=None) -> list:
        """
        Two-step ReAct pipeline over a SET of concepts (which may overlap):
        plan one unified question list across all concepts, then design each one.
        Each question is anchored to a specific content_id from `contents`.
        Returns a list of question dicts ready for the study presenter.
        """
        _log = log_fn or print
        if not contents:
            return []

        _log('\n🧠 ReAct — Step 1/2: Planning unified question set across concepts…')
        plan = self._run_plan_step(contents, log_fn)
        if not plan:
            _log('  ⚠  No questions planned.')
            return []
        _log(f'  ✓ {len(plan)} question(s) planned:')
        title_by_id = {c.id: c.title for c in contents}
        for i, p in enumerate(plan, 1):
            qt = p['question_type'].upper()
            qs = p.get('question_style', '')
            label = qt if qt == 'CODE_DEMO' else f'{qt} / {qs}'
            anchor = title_by_id.get(p.get('content_id', ''), '?')
            _log(f'     {i}. [{label}]  ⟶  {anchor}\n         {p["focus"]}')

        _log('\n🧠 ReAct — Step 2/2: Designing each question…')
        questions = self._run_design_step(contents, plan, log_fn)
        _log(f'  ✓ {len(questions)} question(s) ready.')
        return questions

    # ── Step 1: Plan ──────────────────────────────────────────────────────────

    def _run_plan_step(self, contents: list, log_fn=None) -> list:
        """Agent reads ALL concepts and drafts ONE plan, deduping overlap across concepts."""
        result = self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a question planning agent.\n'
                        'Task: read a SET of related learning concepts (which may overlap)\n'
                        'and draft ONE unified plan of questions to design across them.\n\n'
                        'Possible question types (use the exact type key in your plan):\n'
                        '  mcq  / theory    — multiple-choice on conceptual understanding\n'
                        '  open / theory    — student explains in their own words\n'
                        '  mcq  / practical — scenario-based multiple-choice\n'
                        '  open / practical — non-code application question\n'
                        '  code_demo        — YOU use code_interpreter to craft a creative\n'
                        '                     programming question (predict output, find a bug,\n'
                        '                     explain surprising behavior, fill a blank, spot\n'
                        '                     a difference, trace execution). Student writes NO code.\n'
                        '                     Set question_style to "" for code_demo entries.\n\n'
                        'Rules:\n'
                        '  - Treat the set holistically: if two concepts overlap, do NOT plan\n'
                        '    duplicate questions for the same idea — plan it once and anchor\n'
                        '    it to whichever concept covers it best.\n'
                        '  - For each planned question, set `content_id` to the SINGLE concept\n'
                        '    it primarily anchors. Use only ids from the provided list.\n'
                        '  - Prefer breadth across concepts over many questions on one concept.\n'
                        '  - For each planned question write a clear `focus`.\n'
                        '  - Prefer code_demo over open/practical when Python/SQL applies.\n'
                        '  - Never plan a question that asks the student to write code.\n'
                        '  - Do not write the actual questions yet — just the plan.\n'
                        '  - Call submit_plan when your plan is complete.'
                    ),
                },
                {
                    'role': 'user',
                    'content': (
                        f'{self._format_concepts(contents)}\n\n'
                        'Draft ONE unified question plan for this set of concepts.'
                    ),
                },
            ],
            tools=[self._plan_tool(contents)],
            final_tool_name='submit_plan',
            execute_fn=lambda name, args: 'Plan received.',
            step_name='plan',
            log_fn=log_fn,
            show_reasoning=True,
        )
        valid_ids = {c.id for c in contents}
        plan = []
        for entry in result.get('questions', []) or []:
            if entry.get('content_id') in valid_ids:
                plan.append(entry)
        return plan

    # ── Step 2: Design ────────────────────────────────────────────────────────

    def _run_design_step(self, contents: list, plan: list, log_fn=None) -> list:
        """Agent takes the plan and designs each question in full detail across all concepts."""
        state     = {'questions': []}
        valid_ids = {c.id for c in contents}
        title_by_id = {c.id: c.title for c in contents}

        def _resolve_content_id(args):
            cid = args.get('content_id', '')
            return cid if cid in valid_ids else (contents[0].id if contents else '')

        def execute_fn(name, args):
            if name == 'add_mcq_question':
                q = {k: v for k, v in args.items() if k != 'thought'}
                q['question_type'] = 'mcq'
                q['content_id']    = _resolve_content_id(args)
                state['questions'].append(q)
                return f'MCQ added ({q["question_style"]}, ⟶ {title_by_id.get(q["content_id"], "?")}). Total: {len(state["questions"])}'
            if name == 'add_open_question':
                q = {k: v for k, v in args.items() if k != 'thought'}
                q['question_type'] = 'open'
                q['language']      = 'none'
                q['test_code']     = ''
                q['content_id']    = _resolve_content_id(args)
                state['questions'].append(q)
                return f'Open added ({q["question_style"]}, ⟶ {title_by_id.get(q["content_id"], "?")}). Total: {len(state["questions"])}'
            if name == 'add_code_question':
                q = {
                    'question_type':  'open',
                    'question_style': args.get('question_style', 'practical'),
                    'language':       'none',
                    'question':       args.get('question', ''),
                    'model_answer':   args.get('model_answer', ''),
                    'rubric':         args.get('rubric', ''),
                    'content_id':     _resolve_content_id(args),
                }
                state['questions'].append(q)
                return f'Code demo question added (⟶ {title_by_id.get(q["content_id"], "?")}). Total: {len(state["questions"])}'
            if name == 'skip_question':
                idx = args.get('plan_index', '?')
                return f'Skipped planned question {idx}: {args.get("reason", "")}'
            return f'Unknown tool: {name}'

        def _plan_label(p):
            qt = p['question_type'].upper()
            qs = p.get('question_style', '')
            return qt if qt == 'CODE_DEMO' else f'{qt} / {qs}'
        plan_text = '\n'.join(
            f'  {i+1}. [{_plan_label(p)}] (anchor: {title_by_id.get(p.get("content_id",""), "?")})  {p["focus"]}'
            for i, p in enumerate(plan)
        )

        self._react_loop(
            initial_input=[
                {
                    'role': 'system',
                    'content': (
                        'You are a question design agent.\n'
                        'Task: design each question from the plan in full detail.\n\n'
                        'Rules:\n'
                        '  - Work through the plan in order, one question at a time.\n'
                        '  - For every add_*_question call, set `content_id` to the same\n'
                        '    concept the planner anchored that entry to (or a better-fitting\n'
                        '    one if you have a strong reason). Use only ids from the provided\n'
                        '    set of concepts.\n'
                        '  - Use add_mcq_question for MCQ entries.\n'
                        '  - Use add_open_question for text-based open entries ONLY — no code.\n'
                        '    The student will type a written answer; do not ask them to write code.\n'
                        '  - For Code Demo entries, use code_interpreter as your scratchpad:\n'
                        '    Choose one of these creative question styles:\n'
                        '      predict_output   — show code WITHOUT the output; ask "What does this print?"\n'
                        '                         Run CI to verify YOUR expected answer is correct.\n'
                        '      find_bug         — show intentionally broken/tricky code; ask what is wrong.\n'
                        '                         Run CI to confirm the bug/error actually occurs.\n'
                        '      explain_behavior — show code + its surprising output; ask why it behaves that way.\n'
                        '                         Run CI to capture the real output to include in the question.\n'
                        '      fill_blank       — show code with a ___ or ???; ask what expression/value fits.\n'
                        '                         Run CI with the correct fill to confirm it works.\n'
                        '      spot_difference  — show two near-identical snippets; ask how their outputs differ.\n'
                        '                         Run CI on both to understand the difference yourself first.\n'
                        '      trace_execution  — ask what a specific variable equals at a certain point.\n'
                        '                         Run CI with print statements to verify the trace.\n'
                        '    Rules for code questions:\n'
                        '      - Write the FULL question text in the `question` field — you control the format.\n'
                        '      - Include code in ```python or ```sql fences within your question text.\n'
                        '      - Reveal only what the student should see (omit output for predict/find_bug).\n'
                        '      - You may also call add_mcq_question with a code-containing stem for MCQ variants.\n'
                        '      - Student writes NO code.\n'
                        '  - If a planned question cannot be designed, call skip_question.\n'
                        '  - Focus on understanding and application, not memorisation.\n'
                        '  - Call submit_question_set when all planned questions are done.'
                    ),
                },
                {
                    'role': 'user',
                    'content': (
                        f'{self._format_concepts(contents)}\n\n'
                        f'Plan to execute:\n{plan_text}\n\n'
                        'Design each planned question now, in order.'
                    ),
                },
            ],
            tools=[{'type': 'code_interpreter', 'container': {'type': 'auto'}}] + self._design_tools(contents),
            final_tool_name='submit_question_set',
            execute_fn=execute_fn,
            step_name='design',
            log_fn=log_fn,
            show_reasoning=True,
        )
        return state['questions']

    @staticmethod
    def _format_concepts(contents: list) -> str:
        """Render the set of concepts as a labelled block for prompts."""
        blocks = []
        for c in contents:
            blocks.append(f'[id={c.id}] {c.title}\n{c.body}')
        return 'Concepts in this study set (overlap may exist):\n\n' + '\n\n---\n\n'.join(blocks)

    def generate_followup(self, content: Content, prior_question: dict,
                          student_answer: str, score: float) -> Optional[dict]:
        """Return a follow-up open-ended question, or None if not warranted."""
        fmt_hint = 'open-ended JSON: {"question":"...","language":"none","model_answer":"...","rubric":"...","test_code":""}'
        response = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'Concept: {content.title}\n{content.body}\n\n'
                f'The student was asked:\n{prior_question.get("question", "")}\n\n'
                f'Their answer: {student_answer}\nScore: {score:.1f}/1.0\n\n'
                'Based on the concept, the question, and the student\'s answer: '
                'is there a meaningful gap in understanding that a follow-up question would help address? '
                'Only ask a follow-up if it genuinely deepens learning — do not ask for the sake of it.\n'
                f'If yes, write one follow-up open-ended question as {fmt_hint}.\n'
                'If no follow-up is needed, output exactly: NO_FOLLOWUP'
            )
        )
        raw = response.output_text.strip()
        if 'NO_FOLLOWUP' in raw:
            return None
        try:
            q = Ingester._parse_json(raw)
            q['content_id']     = content.id
            q['question_type']  = 'open'
            q['question_style'] = prior_question.get('question_style', 'theory')
            return q
        except Exception:
            return None

    # ── Tool definitions ───────────────────────────────────────────────────────

    def _plan_tool(self, contents: list) -> dict:
        content_ids = [c.id for c in contents]
        return self._fn_tool(
            'submit_plan',
            'Submit the unified question plan across the provided concept set. '
            'Each entry says what type to design, what it should test, and which '
            'concept it primarily anchors to.',
            {
                'thought': {
                    'type': 'string',
                    'description': 'Overall reasoning: which angles are worth testing across the concept set, and how you avoided overlap.',
                },
                'questions': {
                    'type': 'array',
                    'description': 'Ordered list of questions to design across the whole set.',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'question_type':  {'type': 'string', 'enum': ['mcq', 'open', 'code_demo']},
                            'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                            'content_id':     {
                                'type': 'string',
                                'enum': content_ids,
                                'description': 'The concept this question primarily anchors to. Must be one of the provided ids.',
                            },
                            'focus': {
                                'type': 'string',
                                'description': 'What this question tests and why it aids learning.',
                            },
                        },
                        'required': ['question_type', 'question_style', 'content_id', 'focus'],
                        'additionalProperties': False,
                    },
                },
            },
            ['thought', 'questions'],
        )

    def _design_tools(self, contents: list) -> list:
        content_ids = [c.id for c in contents]
        return [
            self._fn_tool(
                'add_mcq_question',
                'Add a multiple-choice question (theory or practical) to the set.',
                {
                    'thought':        {'type': 'string', 'description': 'Why this MCQ is valuable and well-crafted for this concept.'},
                    'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                    'content_id':     {'type': 'string', 'enum': content_ids, 'description': 'The concept this question anchors to.'},
                    'question':       {'type': 'string', 'description': 'The question stem.'},
                    'options':        {'type': 'array',  'items': {'type': 'string'}, 'description': 'Exactly 4 answer options, no A/B/C/D prefix.'},
                    'correct_index':  {'type': 'integer', 'description': '0-based index of the correct option.'},
                    'explanation':    {'type': 'string',  'description': 'Why the correct answer is right and the distractors are wrong.'},
                },
                ['thought', 'question_style', 'content_id', 'question', 'options', 'correct_index', 'explanation'],
            ),
            self._fn_tool(
                'add_open_question',
                'Add a text-based open-ended question (theory or practical). The student types a written answer — no code.',
                {
                    'thought':        {'type': 'string', 'description': 'Why this open question is valuable for this concept.'},
                    'question_style': {'type': 'string', 'enum': ['theory', 'practical']},
                    'content_id':     {'type': 'string', 'enum': content_ids, 'description': 'The concept this question anchors to.'},
                    'question':       {'type': 'string', 'description': 'The question text.'},
                    'model_answer':   {'type': 'string', 'description': 'Ideal written answer.'},
                    'rubric':         {'type': 'string', 'description': 'Grading criteria — what a good answer must cover.'},
                },
                ['thought', 'question_style', 'content_id', 'question', 'model_answer', 'rubric'],
            ),
            self._fn_tool(
                'add_code_question',
                'Add a creative programming question. Use code_interpreter first to '
                'verify behavior, craft distractors, or confirm a bug — then call '
                'this with the complete question the student will see. Student writes NO code.',
                {
                    'thought': {
                        'type': 'string',
                        'description': 'How you used code_interpreter to craft this question and why this style suits the concept.',
                    },
                    'question_style': {
                        'type': 'string',
                        'enum': ['predict_output', 'find_bug', 'explain_behavior', 'fill_blank', 'spot_difference', 'trace_execution'],
                        'description': (
                            'predict_output: show code, ask what it prints. '
                            'find_bug: show broken code, ask what is wrong. '
                            'explain_behavior: show surprising output, ask why. '
                            'fill_blank: show code with a gap, ask what goes there. '
                            'spot_difference: show two snippets, ask how their outputs differ. '
                            'trace_execution: ask what a variable equals at a specific point.'
                        ),
                    },
                    'content_id': {'type': 'string', 'enum': content_ids, 'description': 'The concept this question anchors to.'},
                    'question': {
                        'type': 'string',
                        'description': (
                            'Complete question text shown to the student. '
                            'Include code in ```python or ```sql fences. '
                            'You control exactly what to show — omit the real output for '
                            'predict/find_bug, include surprising output for explain_behavior, etc. '
                            'End with a clear question sentence.'
                        ),
                    },
                    'model_answer': {'type': 'string', 'description': 'Correct answer a strong student would give.'},
                    'rubric':       {'type': 'string', 'description': 'Grading criteria — what must a good answer cover.'},
                },
                ['thought', 'question_style', 'content_id', 'question', 'model_answer', 'rubric'],
            ),
            self._fn_tool(
                'skip_question',
                'Skip a planned question that cannot be designed as intended.',
                {
                    'thought':      {'type': 'string'},
                    'plan_index':   {'type': 'integer', 'description': '1-based index of the planned question to skip.'},
                    'reason':       {'type': 'string', 'description': 'Why this question cannot be designed as planned.'},
                },
                ['thought', 'plan_index', 'reason'],
            ),
            self._fn_tool(
                'submit_question_set',
                'Finalise the question set after designing all planned questions.',
                {'thought': {'type': 'string', 'description': 'Summary of the designed set.'}},
                ['thought'],
            ),
        ]


content_retriever = ContentRetriever(client, store, vector_store, content_manager)
question_engine   = QuestionEngine(client)
print('✓ ContentRetriever + QuestionEngine ready')


✓ ContentRetriever + QuestionEngine ready


## Cell 10 — Answer Evaluation (`Grader`)

In [10]:
class Grader:
    """Evaluates student answers for MCQ, open-ended, and Python/SQL code questions."""

    def __init__(self, openai_client):
        self.client = openai_client

    def grade(self, question: dict, student_answer: str) -> dict:
        """Route to the correct grader. Returns {score, correct, feedback}."""
        q_type   = question.get('question_type', 'open')
        language = question.get('language', 'none')
        if q_type == 'mcq':
            return self._grade_mcq(question, student_answer)
        if language in ('python', 'sql'):
            return self._grade_code(question, student_answer)
        return self._grade_open(question, student_answer)

    # ── MCQ (deterministic) ───────────────────────────────────────────────────

    def _grade_mcq(self, question: dict, student_answer: str) -> dict:
        try:
            chosen = int(student_answer)
        except (ValueError, TypeError):
            chosen = -1
        correct_idx = question.get('correct_index', -1)
        is_correct  = (chosen == correct_idx)
        explanation = question.get('explanation', '')
        return {
            'score':    1.0 if is_correct else 0.0,
            'correct':  is_correct,
            'feedback': (
                f'✅ Correct! {explanation}'
                if is_correct
                else f'❌ Incorrect. Right answer was option {correct_idx + 1}. {explanation}'
            )
        }

    # ── Open-ended (GPT-4o rubric) ────────────────────────────────────────────

    def _grade_open(self, question: dict, student_answer: str) -> dict:
        resp = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'Question: {question.get("question", "")}\n\n'
                f'Model answer: {question.get("model_answer", "")}\n\n'
                f'Rubric: {question.get("rubric", "")}\n\n'
                f'Student answer: {student_answer}\n\n'
                'Grade 0.0–1.0. Be encouraging but honest. '
                'Output ONLY valid JSON:\n'
                '{"score": 0.0, "correct": false, "feedback": "what was correct / missing / wrong"}'
            )
        )
        result = Ingester._parse_json(resp.output_text)
        result['correct'] = result.get('score', 0) >= 0.6
        return result

    # ── Code — Python / SQL (Code Interpreter) ────────────────────────────────

    def _grade_code(self, question: dict, student_answer: str) -> dict:
        language  = question.get('language', 'python')
        test_code = question.get('test_code', '')

        if language == 'sql':
            combined = (
                'import sqlite3\n'
                'conn = sqlite3.connect(":memory:")\n'
                'cur  = conn.cursor()\n\n'
                f'# --- Student SQL ---\n{student_answer}\n\n'
                f'# --- Tests ---\n{test_code}'
            )
        else:
            combined = f'# --- Student code ---\n{student_answer}\n\n# --- Tests ---\n{test_code}'

        try:
            resp = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'code_interpreter', 'container': {'type': 'auto'}}],
                input=(
                    f'Execute this {language.upper()} code and report whether all tests pass.\n\n'
                    f'```python\n{combined}\n```\n\n'
                    'After execution output ONLY valid JSON:\n'
                    '{"score": 1.0, "correct": true, "feedback": "...", "execution_output": "..."}'
                )
            )
            return Ingester._parse_json(resp.output_text)
        except Exception:
            return self._grade_open(question, student_answer)


grader = Grader(client)
print('✓ Grader ready')

✓ Grader ready


## Cell 11 — Study Session (`StudySession`)

In [11]:
class StudySession:
    """Provides content recommendations for a study session."""

    def __init__(self, store: Store, vector_store: VectorStore):
        self.store        = store
        self.vector_store = vector_store

    def recommend_content(self, progress_tracker, n: int = 5) -> list:
        """
        Priority:
        1. Weak content (avg score < 0.6 over last 5 attempts)
        2. Semantically related to recent sessions
        3. Random sample of all content
        """
        # 1. Weak topics
        weak = progress_tracker.get_weak_content_ids()
        if weak:
            return weak[:n]

        # 2. Semantic similarity to recent sessions
        recent = progress_tracker.get_recent_sessions(n=3)
        if recent:
            titles = [
                c.title
                for s in recent
                if (c := self.store.get_content(s.content_id)) is not None
            ]
            if titles:
                found = self.vector_store.search(' '.join(titles), top_k=n)
                if found:
                    return found

        # 3. Random fallback
        all_ids = [c.id for c in self.store.all_content()]
        return random.sample(all_ids, min(n, len(all_ids)))


study_session = StudySession(store, vector_store)
print('✓ StudySession ready')

✓ StudySession ready


## Cell 12 — Progress Tracking (`ProgressTracker`)

In [12]:
class ProgressTracker:
    """Tracks session history, weak topics, and study streaks."""

    WEAK_THRESHOLD = 0.6
    WEAK_WINDOW    = 5

    def __init__(self, store: Store):
        self.store = store

    def record(self, record: SessionRecord):
        self.store.sessions.append(record)
        self._recompute_weak()
        self.store._save_progress()

    def get_weak_content_ids(self) -> list:
        return list(self.store.weak_content_ids)

    def get_recent_sessions(self, n: int = 10) -> list:
        return self.store.sessions[-n:]

    def get_study_streak(self) -> int:
        """Count consecutive calendar days with at least one session."""
        if not self.store.sessions:
            return 0
        days   = sorted({s.date[:10] for s in self.store.sessions}, reverse=True)
        streak = 1
        for i in range(1, len(days)):
            delta = (date.fromisoformat(days[i - 1]) - date.fromisoformat(days[i])).days
            if delta == 1:
                streak += 1
            else:
                break
        return streak

    def summary(self) -> dict:
        recent = self.store.sessions[-10:]
        avg    = sum(s.score for s in recent) / len(recent) if recent else 0.0
        weak_titles = [
            c.title
            for cid in self.store.weak_content_ids
            if (c := self.store.get_content(cid)) is not None
        ]
        return {
            'total_content':    len(self.store.all_content()),
            'total_topics':     len(self.store.all_topics()),
            'total_sessions':   len(self.store.sessions),
            'avg_score_last10': avg,
            'weak_topics':      weak_titles,
            'streak_days':      self.get_study_streak(),
        }

    def get_content_performance(self, content_id: str) -> dict:
        """Return latest score, window avg, last attempt date, and total attempts for one content item."""
        sessions = [s for s in self.store.sessions if s.content_id == content_id]
        if not sessions:
            return {'latest_score': None, 'avg_score': None, 'latest_date': None, 'attempts': 0}
        latest   = sessions[-1]
        window   = sessions[-self.WEAK_WINDOW:]
        avg      = sum(s.score for s in window) / len(window)
        return {
            'latest_score': latest.score,
            'avg_score':    avg,
            'latest_date':  latest.date[:10],
            'attempts':     len(sessions),
        }

    def _recompute_weak(self):
        scores_by: dict = {}
        for s in self.store.sessions:
            scores_by.setdefault(s.content_id, []).append(s.score)
        self.store.weak_content_ids = [
            cid
            for cid, scores in scores_by.items()
            if (sum(scores[-self.WEAK_WINDOW:]) / min(len(scores), self.WEAK_WINDOW))
               < self.WEAK_THRESHOLD
        ]


progress_tracker = ProgressTracker(store)
print('✓ ProgressTracker ready')

✓ ProgressTracker ready


## Cell 13 — Content Management Agent (`ManagementAgent`)

In [13]:
class ManagementAgent(Agent):
    """ReAct agent for content management: list, create, rename, move, and delete topics/content."""

    SYSTEM_PROMPT = (
        'You are a content-management assistant with tools to inspect and modify a topic/content tree.\n'
        'Work step-by-step: use list tools first when you need to resolve ambiguous names or verify state,\n'
        'then call the appropriate action tools one at a time.\n'
        'When the user\'s request requires multiple actions (e.g. "remove all empty topics"), \n'
        'iterate — call each action, observe the result, then continue until the goal is fully achieved.\n'
        'When `show_content` is called, the content is printed directly to the user terminal — \n'
        'do NOT repeat the full body in the `finish` message; just confirm what was shown.\n'
        '\n'
        'MANDATORY RULE — Before creating any new content item:\n'
        '  1. Call `search_content` with the new concept title/description as the query (top_k=3).\n'
        '  2. If any results are returned, call `show_content` on the closest match.\n'
        '  3. Decide: if the existing content covers substantially the same concept, call `update_content`\n'
        '     to merge the new material into the existing item (update both title and body as needed),\n'
        '     then skip `create_content`. Otherwise proceed with `create_content` as normal.\n'
        '  4. Always explain your merge/create decision in the `thought` field.\n'
        '\n'
        'When done, call `finish` with a concise, friendly summary of everything that was accomplished.\n'
        'If a request is impossible (e.g. deleting root, circular move, target not found), \n'
        'think carefully about why, then call `finish` explaining the reason clearly.'
    )

    _TOOLS = [
              ToolDefs.LIST_TREE,
        Agent._fn_tool(
            'list_topics',
            'Return all topics as JSON with id, name, parent_id, and children_count.',
            {},
        ),
        Agent._fn_tool(
            'list_content',
            'Return all content items as JSON with id, title, and topic_id.',
            {},
        ),
              ToolDefs.CREATE_TOPIC,
        Agent._fn_tool(
            'rename_topic',
            'Rename an existing topic.',
            {
                'thought':   {'type': 'string'},
                'topic_ref': {'type': 'string', 'description': 'Name or ID prefix of topic to rename.'},
                'new_name':  {'type': 'string', 'description': 'New name.'},
            },
            required=['thought', 'topic_ref', 'new_name'],
        ),
        Agent._fn_tool(
            'rename_content',
            'Rename (retitle) a content item.',
            {
                'thought':     {'type': 'string'},
                'content_ref': {'type': 'string', 'description': 'Name or ID prefix of content to rename.'},
                'new_title':   {'type': 'string', 'description': 'New title.'},
            },
            required=['thought', 'content_ref', 'new_title'],
        ),
        Agent._fn_tool(
            'move_topic',
            'Move a topic under a new parent. Cannot move root or into a descendant.',
            {
                'thought':        {'type': 'string'},
                'topic_ref':      {'type': 'string', 'description': 'Name or ID prefix of topic to move.'},
                'new_parent_ref': {'type': 'string', 'description': 'Name or ID prefix of new parent topic.'},
            },
            required=['thought', 'topic_ref', 'new_parent_ref'],
        ),
        Agent._fn_tool(
            'move_content',
            'Move a content item to a different topic.',
            {
                'thought':     {'type': 'string'},
                'content_ref': {'type': 'string', 'description': 'Name or ID prefix of content to move.'},
                'topic_ref':   {'type': 'string', 'description': 'Name or ID prefix of destination topic.'},
            },
            required=['thought', 'content_ref', 'topic_ref'],
        ),
        Agent._fn_tool(
            'delete_topic',
            'Delete a topic. Its child topics and content are reparented to its parent. Cannot delete root.',
            {
                'thought':   {'type': 'string'},
                'topic_ref': {'type': 'string', 'description': 'Name or ID prefix of topic to delete.'},
            },
            required=['thought', 'topic_ref'],
        ),
        Agent._fn_tool(
            'delete_content',
            'Delete a content item permanently.',
            {
                'thought':     {'type': 'string'},
                'content_ref': {'type': 'string', 'description': 'Name or ID prefix of content to delete.'},
            },
            required=['thought', 'content_ref'],
        ),
              ToolDefs.SEARCH_CONTENT,
        Agent._fn_tool(
            'create_content',
            (
                'Create a brand-new content item in the tree and index it in the vector store. '
                'Only call this AFTER searching for similar content and deciding not to merge.'
            ),
            {
                'thought':   {'type': 'string', 'description': 'Reasoning — why no merge is appropriate.'},
                'title':     {'type': 'string', 'description': 'Title of the new content item.'},
                'body':      {'type': 'string', 'description': 'Full body text of the concept.'},
                'topic_ref': {'type': 'string', 'description': 'Name or ID prefix of the target topic.'},
            },
            required=['thought', 'title', 'body', 'topic_ref'],
        ),
        Agent._fn_tool(
            'update_content',
            (
                'Update an existing content item (title and/or body) and re-index it in the vector store. '
                'Use this to merge new material into an existing concept.'
            ),
            {
                'thought':     {'type': 'string', 'description': 'Reasoning — what is being merged and why.'},
                'content_ref': {'type': 'string', 'description': 'Name or ID prefix of the content to update.'},
                'new_title':   {'type': 'string', 'description': 'Updated title; pass the existing title to leave it unchanged.'},
                'new_body':    {'type': 'string', 'description': 'Updated/merged body text.'},
            },
            required=['thought', 'content_ref', 'new_title', 'new_body'],
        ),
        Agent._fn_tool(
            'show_content',
            'Display the full title and body of one or more content items.',
            {
                'thought':      {'type': 'string'},
                'content_refs': {
                    'type': 'array',
                    'items': {'type': 'string'},
                    'description': 'List of content names or ID prefixes to display.',
                },
            },
            required=['thought', 'content_refs'],
        ),
        Agent._fn_tool(
            'finish',
            'End the task and return the final message to the user.',
            {
                'thought': {'type': 'string'},
                'message': {'type': 'string', 'description': 'User-facing result or explanation.'},
            },
            required=['thought', 'message'],
        ),
    ]

    def __init__(self, openai_client, content_manager, vector_store):
        super().__init__(openai_client)
        self.cm = content_manager
        self.vs = vector_store

    # ── Public entry point ────────────────────────────────────────────────────

    def run(self, command: str, log_fn=None, show_reasoning: bool = True) -> str:
        """Execute a natural-language management command; return a user-facing result string."""
        st = self.cm.store
        topic_state   = [{'id': t.id, 'name': t.name, 'parent_id': t.parent_id,
                           'children_count': len(t.children_topic_ids),
                           'content_count': len(t.content_ids)}
                          for t in st.all_topics()]
        content_state = [{'id': c.id, 'title': c.title, 'topic_id': c.topic_id}
                          for c in st.all_content()]

        initial_input = [
            {
                'role': 'system',
                'content': (
                    self.SYSTEM_PROMPT + '\n\n'
                    f'Current topics (JSON): {json.dumps(topic_state)}\n'
                    f'Current content (JSON): {json.dumps(content_state)}'
                ),
            },
            {'role': 'user', 'content': command},
        ]

        result = self._react_loop(
            initial_input=initial_input,
            tools=self._TOOLS,
            final_tool_name='finish',
            execute_fn=self._dispatch,
            step_name='manage',
            log_fn=log_fn,
            show_reasoning=show_reasoning,
        )
        return result.get('message', '⚠ Agent did not return a message.')

    # ── Resolve helpers ───────────────────────────────────────────────────────

    @staticmethod
    def _normalize(text):
        return (text or '').strip().lower()

    def _topic_children_ids(self, tid):
        topic = self.cm.store.get_topic(tid)
        if not topic:
            return []
        out = []
        for cid in topic.children_topic_ids:
            out.append(cid)
            out.extend(self._topic_children_ids(cid))
        return out

    def _resolve_topic(self, ref):
        ref = (ref or '').strip()
        if not ref:
            return None
        topics = self.cm.store.all_topics()
        exact_id = next((t for t in topics if t.id == ref), None)
        if exact_id:
            return exact_id
        prefix_id = next((t for t in topics if t.id.startswith(ref)), None)
        if prefix_id:
            return prefix_id
        key = self._normalize(ref)
        exact_name = [t for t in topics if self._normalize(t.name) == key]
        if len(exact_name) == 1:
            return exact_name[0]
        by_contains = [t for t in topics if key in self._normalize(t.name)]
        return by_contains[0] if len(by_contains) == 1 else None

    def _resolve_content(self, ref):
        ref = (ref or '').strip()
        if not ref:
            return None
        content = self.cm.store.all_content()
        exact_id = next((c for c in content if c.id == ref), None)
        if exact_id:
            return exact_id
        prefix_id = next((c for c in content if c.id.startswith(ref)), None)
        if prefix_id:
            return prefix_id
        key = self._normalize(ref)
        exact_title = [c for c in content if self._normalize(c.title) == key]
        if len(exact_title) == 1:
            return exact_title[0]
        by_contains = [c for c in content if key in self._normalize(c.title)]
        return by_contains[0] if len(by_contains) == 1 else None

    # ── Presentation ─────────────────────────────────────────────────────────

    @staticmethod
    def _print_content(content, topic):
        """Print a single content item directly to the user terminal."""
        print()
        print('─' * 60)
        print(f'  📄  {content.title}')
        print(f'  Topic : {topic.name if topic else "?"}')
        print('─' * 60)
        print()
        for line in (content.body or '').splitlines():
            print(f'   {line}')
        print()

    # ── Tool dispatcher ───────────────────────────────────────────────────────

    def _dispatch(self, name: str, args: dict) -> str:
        st = self.cm.store
        if name == 'list_tree':
            return self.cm.list_topics()

        if name == 'list_topics':
            rows = [{'id': t.id, 'name': t.name, 'parent_id': t.parent_id,
                     'children_count': len(t.children_topic_ids),
                     'content_count':  len(t.content_ids)}
                    for t in st.all_topics()]
            return json.dumps(rows)

        if name == 'list_content':
            rows = [{'id': c.id, 'title': c.title, 'topic_id': c.topic_id}
                    for c in st.all_content()]
            return json.dumps(rows)

        if name == 'create_topic':
            name_val   = (args.get('name') or '').strip()
            parent_ref = (args.get('parent_ref') or '').strip()
            if not name_val:
                return '⚠ name is required.'
            parent = self._resolve_topic(parent_ref) if parent_ref else st.root_topic()
            if not parent:
                return f'⚠ Parent topic not found: {parent_ref!r}'
            t = self.cm.create_topic(name_val, parent.id)
            return f'Created topic "{t.name}" under "{parent.name}" (id={t.id}).'

        if name == 'rename_topic':
            topic    = self._resolve_topic(args.get('topic_ref', ''))
            new_name = (args.get('new_name') or '').strip()
            if not topic:
                return f'⚠ Topic not found: {args.get("topic_ref")!r}'
            if not new_name:
                return '⚠ new_name is required.'
            old = topic.name
            topic.name = new_name
            st._save_tree()
            return f'Renamed topic "{old}" → "{new_name}".'

        if name == 'rename_content':
            c         = self._resolve_content(args.get('content_ref', ''))
            new_title = (args.get('new_title') or '').strip()
            if not c:
                return f'⚠ Content not found: {args.get("content_ref")!r}'
            if not new_title:
                return '⚠ new_title is required.'
            old = c.title
            c.title = new_title
            st._save_tree()
            return f'Renamed content "{old}" → "{new_title}".'

        if name == 'move_topic':
            topic      = self._resolve_topic(args.get('topic_ref', ''))
            new_parent = self._resolve_topic(args.get('new_parent_ref', ''))
            if not topic:
                return f'⚠ Topic not found: {args.get("topic_ref")!r}'
            if not new_parent:
                return f'⚠ Destination topic not found: {args.get("new_parent_ref")!r}'
            root = st.root_topic()
            if root and topic.id == root.id:
                return '⚠ Cannot move root topic.'
            if new_parent.id == topic.id:
                return '⚠ Cannot move a topic under itself.'
            if new_parent.id in self._topic_children_ids(topic.id):
                return '⚠ Cannot move a topic into one of its own descendants.'
            old_parent = st.get_topic(topic.parent_id)
            if old_parent and topic.id in old_parent.children_topic_ids:
                old_parent.children_topic_ids.remove(topic.id)
            topic.parent_id = new_parent.id
            if topic.id not in new_parent.children_topic_ids:
                new_parent.children_topic_ids.append(topic.id)
            st._save_tree()
            return f'Moved topic "{topic.name}" under "{new_parent.name}".'

        if name == 'move_content':
            c = self._resolve_content(args.get('content_ref', ''))
            t = self._resolve_topic(args.get('topic_ref', ''))
            if not c:
                return f'⚠ Content not found: {args.get("content_ref")!r}'
            if not t:
                return f'⚠ Destination topic not found: {args.get("topic_ref")!r}'
            self.cm.move_content(c.id, t.id)
            return f'Moved content "{c.title}" to topic "{t.name}".'

        if name == 'delete_topic':
            topic = self._resolve_topic(args.get('topic_ref', ''))
            if not topic:
                return f'⚠ Topic not found: {args.get("topic_ref")!r}'
            root = st.root_topic()
            if root and topic.id == root.id:
                return '⚠ Cannot delete root topic.'
            parent = st.get_topic(topic.parent_id)
            if not parent:
                return '⚠ Parent topic is missing — cannot delete safely.'
            for child_id in list(topic.children_topic_ids):
                child = st.get_topic(child_id)
                if child:
                    child.parent_id = parent.id
                    if child.id not in parent.children_topic_ids:
                        parent.children_topic_ids.append(child.id)
            for cid in list(topic.content_ids):
                c = st.get_content(cid)
                if c:
                    c.topic_id = parent.id
                    if cid not in parent.content_ids:
                        parent.content_ids.append(cid)
            if topic.id in parent.children_topic_ids:
                parent.children_topic_ids.remove(topic.id)
            del st.topics[topic.id]
            st._save_tree()
            return f'Deleted topic "{topic.name}"; children/content reparented to "{parent.name}".'

        if name == 'delete_content':
            c = self._resolve_content(args.get('content_ref', ''))
            if not c:
                return f'⚠ Content not found: {args.get("content_ref")!r}'
            file_id = self.cm.delete_content(c.id)
            self.vs.delete_content(file_id)
            return f'Deleted content "{c.title}".'

        if name == 'search_content':
            query = (args.get('query') or '').strip()
            top_k = int(args.get('top_k') or 5)
            if not query:
                return '⚠ query is required.'
            ids = self.vs.search(query, top_k=top_k)
            if not ids:
                return 'No matching content found.'
            rows = []
            for cid in ids:
                c = st.get_content(cid)
                if c:
                    topic = st.get_topic(c.topic_id)
                    rows.append({'id': c.id, 'title': c.title,
                                 'topic': topic.name if topic else '?'})
            return json.dumps(rows)

        if name == 'show_content':
            refs = args.get('content_refs') or []
            if not refs:
                return '⚠ content_refs is required.'
            parts = []
            for ref in refs:
                c = self._resolve_content(ref)
                if not c:
                    parts.append(f'⚠ Not found: {ref!r}')
                    continue
                topic = st.get_topic(c.topic_id)
                self._print_content(c, topic)
                parts.append(f'Displayed "{c.title}" to user.')
            return '\n'.join(parts)

        if name == 'create_content':
            title_val = (args.get('title') or '').strip()
            body_val  = (args.get('body') or '').strip()
            topic_ref = (args.get('topic_ref') or '').strip()
            if not title_val:
                return '⚠ title is required.'
            if not body_val:
                return '⚠ body is required.'
            topic = self._resolve_topic(topic_ref) if topic_ref else st.root_topic()
            if not topic:
                return f'⚠ Topic not found: {topic_ref!r}'
            content = Content(
                id=str(uuid.uuid4()),
                title=title_val,
                body=body_val,
                topic_id=topic.id,
                source_file='manual',
                created_at=datetime.utcnow().isoformat(),
            )
            content.vector_file_id = self.vs.add_content(content)
            self.cm.add_content_to_topic(content, topic.id)
            return f'Created content "{content.title}" in topic "{topic.name}" (id={content.id}).'

        if name == 'update_content':
            c         = self._resolve_content(args.get('content_ref', ''))
            new_title = (args.get('new_title') or '').strip() or None
            new_body  = (args.get('new_body') or '').strip() or None
            if not c:
                return f'⚠ Content not found: {args.get("content_ref")!r}'
            if new_body is None:
                return '⚠ new_body is required for update_content.'
            old_file_id = c.vector_file_id
            updated = self.cm.update_content(c.id, new_title=new_title, new_body=new_body)
            if not updated:
                return f'⚠ Failed to update content "{args.get("content_ref")!r}".'
            updated.vector_file_id = self.vs.update_content(updated, old_file_id)
            st._save_tree()
            return (
                f'Updated content "{updated.title}" '
                f'({"title + body" if new_title else "body only"} changed; re-indexed in vector store).'
            )

        return f'⚠ Unknown tool: {name!r}'



print('✓ ManagementAgent defined')

✓ ManagementAgent defined


## Cell 14 — Main Navigation UI

In [14]:
# ── Wire all classes together ──────────────────────────────────────────────────
ingester = Ingester(client, content_manager, vector_store, web_searcher,
                   suggest_similar=ENABLE_WEB_SUGGESTIONS,
                   show_reasoning=True)
management_agent = ManagementAgent(client, content_manager, vector_store)

# ── Presentation helpers ──────────────────────────────────────────────────────
# These run regardless of VERBOSE_LOGGING — they shape the UX, not the agent traces.

UI_WIDTH = 60

def divider(char='─', width=UI_WIDTH):
    print(char * width)

def header(title):
    print()
    divider('═')
    print(f'  {title}')
    divider('═')
    print()

def section(title, char='─'):
    """A lighter sub-divider used inside a screen."""
    print()
    divider(char)
    print(f'  {title}')
    divider(char)

def status(emoji, text):
    """Single-line system notification — always visible, distinct from content."""
    print(f'  {emoji}  {text}')

def indent_block(text, prefix='   '):
    """Indent every line of a multi-line block for readable display."""
    if not text:
        return
    for line in str(text).splitlines() or ['']:
        print(f'{prefix}{line}')

def pause():
    input('  [Enter to continue] ')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MENU
# ══════════════════════════════════════════════════════════════════════════════

def main_menu():
    while True:
        header('🎓  AI Tutor')
        print('  [1] Manage content')
        print('  [2] Ingest new document')
        print('  [3] Study')
        print('  [4] Progress dashboard')
        print('  [0] Exit')
        print()
        c = input('  Choice: ').strip()
        if   c == '1': manage_screen()
        elif c == '2': ingest_screen()
        elif c == '3': study_screen()
        elif c == '4': progress_screen()
        elif c == '0': print('\n  Goodbye!\n'); break
        else:          print('  ⚠  Enter 0–4.')


# ══════════════════════════════════════════════════════════════════════════════
# MANAGE
# ══════════════════════════════════════════════════════════════════════════════

def manage_screen():
    def _help_text():
        return (
            'Natural-language commands are supported. Examples:\n'
            '  "show tree"\n'
            '  "create topic Machine Learning under AI"\n'
            '  "rename topic X to Y"\n'
            '  "move content X to topic Y"\n'
            '  "delete all empty topics"\n'
            '  "remove topic X"'
        )

    while True:
        header('📁  Manage Content')
        print(content_manager.list_topics())
        print()
        print('  Type a command (or "help", "0" to go back):')
        print()
        cmd = input('  > ').strip()
        if cmd == '0':
            break
        if cmd.lower() in ('help', '?'):
            print()
            print(_help_text())
        else:
            print()
            status('💬', f'Command: {cmd}')
            print()
            try:
                result = management_agent.run(cmd, log_fn=agent_log, show_reasoning=VERBOSE_LOGGING)
            except Exception as e:
                result = f'⚠ Agent error: {e}'
            print()
            print(result)
        print()
        pause()

# ══════════════════════════════════════════════════════════════════════════════
# INGEST
# ══════════════════════════════════════════════════════════════════════════════

def ingest_screen():
    if IS_COLAB:
        _ingest_colab()
    else:
        _ingest_local()


def _ingest_colab():
    header('⬆️   Ingest Document')
    print('  A file picker will open — select a PDF or image from your computer.')
    print()
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as exc:
        status('❌', f'Upload failed: {exc}')
        return
    if not uploaded:
        status('⚠ ', 'No file selected.')
        return
    for filename, data in uploaded.items():
        ext = Path(filename).suffix.lower()
        if ext not in Ingester.SUPPORTED_EXTS:
            status('⚠ ', f'Skipping "{filename}" — unsupported type "{ext}".')
            continue
        print()
        status('📥', f'Reading & ingesting "{filename}"… this may take ~30–90s.')
        try:
            ingester.ingest(data, filename, log_fn=agent_log)
            status('✓ ', f'"{filename}" ingested.')
        except Exception as exc:
            status('❌', f'Ingest failed: {exc}')
    print()
    pause()


def _ingest_local():
    header('⬆️   Ingest Document')
    supported = ', '.join(sorted(Ingester.SUPPORTED_EXTS))
    print(f'  Supported: {supported}')
    print()
    while True:
        raw = input('  File path (or 0 to cancel): ').strip().strip('"').strip("'")
        if raw == '0':
            return
        if not raw:
            continue
        path = Path(raw).expanduser().resolve()
        if not path.exists() or not path.is_file():
            status('⚠ ', 'File not found. Try again.')
            continue
        if path.suffix.lower() not in Ingester.SUPPORTED_EXTS:
            status('⚠ ', f'Unsupported type "{path.suffix}". Try again.')
            continue
        break
    print(f'  File : {path.name}  ({path.stat().st_size / 1024:.1f} KB)')
    print()
    status('📥', f'Reading & ingesting "{path.name}"… this may take ~30–90s.')
    try:
        ingester.ingest(path.read_bytes(), path.name, log_fn=agent_log)
        status('✓ ', f'"{path.name}" ingested.')
    except Exception as exc:
        status('❌', f'Ingest failed: {exc}')
    print()
    pause()


# ══════════════════════════════════════════════════════════════════════════════
# STUDY
# ══════════════════════════════════════════════════════════════════════════════

def study_screen():
    while True:
        header('📖  Study')
        print('  [1] Recommended  — system picks weak / related topics')
        print('  [2] Manual       — choose concepts yourself')
        print('  [3] Describe     — type what you want to learn')
        print('  [0] Back')
        print()
        c = input('  Choice: ').strip()
        if c == '0':
            break
        elif c == '1':
            print()
            status('🎯', 'Picking recommended concepts…')
            queue = study_session.recommend_content(progress_tracker)
            if not queue:
                status('⚠ ', 'No content available. Ingest documents first.')
                continue
            # ── Show weak-topic breakdown before starting ──────────────
            weak_ids = progress_tracker.get_weak_content_ids()
            queue_from_weak = [cid for cid in queue if cid in weak_ids]
            if queue_from_weak:
                section('⚠   Weak Topics Detected')
                print()
                print(f'  {"Concept":<35} {"Latest":>7}  {"Avg (last 5)":>12}  {"Attempts":>8}  Last attempt')
                print(f'  {"─"*35:<35} {"─"*6:>7}  {"─"*11:>12}  {"─"*8:>8}  {"─"*12}')
                for cid in queue_from_weak:
                    c_obj = store.get_content(cid)
                    if not c_obj:
                        continue
                    perf  = progress_tracker.get_content_performance(cid)
                    title = (c_obj.title[:33] + '…') if len(c_obj.title) > 34 else c_obj.title
                    latest_s = f"{perf['latest_score']:.0%}" if perf['latest_score'] is not None else 'N/A'
                    avg_s    = f"{perf['avg_score']:.0%}"    if perf['avg_score']    is not None else 'N/A'
                    print(f'  {title:<35} {latest_s:>7}  {avg_s:>12}  {perf["attempts"]:>8}  {perf["latest_date"] or "—"}')
                print()
            else:
                status('✓ ', f'Found {len(queue)} concept(s) to study (no prior weak topics).')
                print()
                for i, cid in enumerate(queue, 1):
                    c_obj = store.get_content(cid)
                    if c_obj:
                        print(f'     {i}. {c_obj.title}')
                print()
            cont = input('  Start session with these concepts? [y/n]: ').strip().lower()
            if cont != 'y':
                continue
            _study_run(queue)
            break
        elif c == '2':
            queue = _study_pick_manual()
            if queue:
                _study_run(queue)
            break
        elif c == '3':
            queue = _study_pick_by_query()
            if queue is _NO_CONTENT:
                print()
                status('⚠ ', 'No matching content in your library.')
                print('      Try rephrasing your request or ingest a relevant document.')
                print()
                pause()
                break
            if not queue:
                continue
            _study_run(queue)
            break
        else:
            status('⚠ ', 'Enter 0, 1, 2, or 3.')


def _study_pick_manual():
    contents = store.all_content()
    if not contents:
        print('  ⚠  No content available. Ingest documents first.')
        return []
    print()
    print('  Available concepts:')
    for i, c in enumerate(contents, 1):
        print(f'    [{i}] {c.title}')
    print()
    raw = input('  Enter numbers separated by commas (e.g. 1,3): ').strip()
    selected = []
    for part in raw.split(','):
        try:
            idx = int(part.strip()) - 1
            if 0 <= idx < len(contents):
                selected.append(contents[idx].id)
        except ValueError:
            pass
    if not selected:
        print('  ⚠  No valid selection.')
    return selected


_NO_CONTENT = object()  # sentinel: retriever found nothing → end study session


def _study_pick_by_query():
    """
    Two-subagent flow:
      1) ContentRetriever picks relevant concepts from the vector store.
      2) Caller passes them to QuestionEngine via _study_run.
    Returns:
      list[str]   — content_ids ready for the question subagent.
      []          — user cancelled at the prompt.
      _NO_CONTENT — retriever returned no matches; end the study session.
    """
    if not store.all_content():
        status('⚠ ', 'No content available. Ingest documents first.')
        return _NO_CONTENT
    print()
    print('  Describe what you want to learn (or 0 to cancel).')
    print('  e.g. "python decorators", "joins in SQL", "transformer attention".')
    print()
    query = input('  > ').strip()
    if not query or query == '0':
        return []

    print()
    status('🔍', f'Searching the library for: {query!r}…')
    try:
        ids = content_retriever.find_relevant_content(query, log_fn=agent_log)
    except Exception as exc:
        status('❌', f'Retrieval error: {exc}')
        return []
    if not ids:
        status('⚠ ', 'No relevant content matched your request.')
        return _NO_CONTENT

    status('✓ ', f'Found {len(ids)} relevant concept(s):')
    for i, cid in enumerate(ids, 1):
        c = store.get_content(cid)
        if c:
            print(f'      {i}. {c.title}')
    return ids


def _study_run(queue):
    contents = [c for c in (store.get_content(cid) for cid in queue) if c]
    if not contents:
        status('⚠ ', 'No valid content in queue.')
        return

    header('📐  Designing Question Set')
    print(f'  Concepts in this set ({len(contents)}):')
    for c in contents:
        print(f'    • {c.title}')
    print()

    status('📝', 'Designing questions… this may take ~30–60s.')
    try:
        questions = question_engine.design_question_set(contents, log_fn=agent_log)
    except Exception as exc:
        status('❌', f'Design error: {exc}')
        return

    if not questions:
        status('⚠ ', 'No questions designed.')
        return

    status('✓ ', f'Question set ready — {len(questions)} question(s).')
    print()

    fallback_content = contents[0]
    all_scores = []
    q_idx      = 0
    fu_count   = 0
    cur_q      = None

    while q_idx < len(questions):
        if cur_q is None:
            cur_q = questions[q_idx]

        content_obj = store.get_content(cur_q.get('content_id', '')) or fallback_content
        q_type   = cur_q.get('question_type', 'open')
        q_style  = cur_q.get('question_style', 'theory')
        q_text   = cur_q.get('question', '')
        suffix   = f'  (follow-up {fu_count})' if fu_count else ''
        title    = f'📖  Question {q_idx + 1} of {len(questions)}{suffix}'

        # ── Question header ────────────────────────────────────────────────
        print()
        divider('═')
        print(f'  {title}')
        print(f'  ⟶  Concept: {content_obj.title}')
        print(f'  📋  Format: {q_type.upper()}  •  Style: {q_style.title()}')
        divider('═')

        # ── Question body ──────────────────────────────────────────────────
        print()
        print('  ❓ Question:')
        print()
        indent_block(q_text, prefix='     ')
        print()

        if q_type == 'mcq':
            options = cur_q.get('options', [])
            print('  📋 Options:')
            print()
            for i, opt in enumerate(options):
                print(f'     {chr(65 + i)}) {opt}')
            print()
            raw_ans = input('  ✏️   Your answer (A/B/C/D): ').strip().upper()
            student_answer = str(ord(raw_ans[0]) - ord('A')) if raw_ans and raw_ans[0] in 'ABCD' else '-1'
            print()
            if student_answer != '-1':
                idx = int(student_answer)
                chosen_opt = options[idx] if 0 <= idx < len(options) else '(invalid)'
                status('➜ ', f'You chose: {chr(65 + idx)}) {chosen_opt}')
            else:
                status('➜ ', 'No valid choice entered.')
        else:
            student_answer = input('  ✏️   Your answer: ').strip()
            print()
            print('  ➜  Your answer:')
            print()
            indent_block(student_answer or '(blank)', prefix='     ')

        # ── Grade ──────────────────────────────────────────────────────────
        print()
        status('⏳', 'Grading…')
        result  = grader.grade(cur_q, student_answer)
        score   = result.get('score', 0.0)
        verdict = '✅  Correct!' if score >= 0.6 else '❌  Needs review'
        all_scores.append(score)

        progress_tracker.record(SessionRecord(
            date=datetime.utcnow().isoformat(),
            content_id=content_obj.id,
            question_type=q_type,
            correct=result.get('correct', score >= 0.6),
            score=score,
        ))

        section(f'Score: {score:.0%}   {verdict}')
        feedback = result.get('feedback', '').strip()
        if feedback:
            print()
            print('  💬 Feedback:')
            print()
            indent_block(feedback, prefix='     ')

        avg = sum(all_scores) / len(all_scores)
        print()
        status('📊', f'Session: {len(all_scores)} answered  •  avg {avg:.0%}')

        # ── Follow-up consideration ────────────────────────────────────────
        fu = None
        if q_type == 'open' and fu_count < QuestionEngine.MAX_FOLLOWUPS and score < 0.9:
            print()
            status('🤔', 'Considering follow-up…')
            fu = question_engine.generate_followup(content_obj, cur_q, student_answer, score)
            if fu:
                status('✓ ', 'A follow-up question is queued.')
            else:
                status('—', 'No follow-up needed.')

        # ── Navigation ─────────────────────────────────────────────────────
        print()
        print('  [1] Next question     [0] Quit session')
        print()
        nav = input('  Choice: ').strip()
        if nav == '0':
            _study_complete(all_scores)
            return

        if fu:
            fu_count += 1
            cur_q = fu
        else:
            q_idx   += 1
            fu_count = 0
            cur_q    = None

    _study_complete(all_scores)


def _study_complete(all_scores):
    avg = sum(all_scores) / len(all_scores) if all_scores else 0.0
    header('🎉  Session Complete!')
    print(f'  Questions answered : {len(all_scores)}')
    print(f'  Average score      : {avg:.0%}')
    print()
    if avg >= 0.8:
        print('  Great work! Keep it up.')
    elif avg >= 0.6:
        print('  Good effort. Review flagged topics and try again.')
    else:
        print('  Several weak areas — recommended session will focus on them.')
    print()


# ══════════════════════════════════════════════════════════════════════════════
# PROGRESS
# ══════════════════════════════════════════════════════════════════════════════

def progress_screen():
    s      = progress_tracker.summary()
    recent = progress_tracker.get_recent_sessions(10)

    header('📊  Progress Dashboard')
    print(f'  Content items  : {s["total_content"]}')
    print(f'  Topics         : {s["total_topics"]}')
    print(f'  Total sessions : {s["total_sessions"]}')
    print(f'  Avg (last 10)  : {s["avg_score_last10"]:.0%}')
    print(f'  Study streak   : {s["streak_days"]} day(s)')
    divider()
    weak = s['weak_topics']
    if weak:
        print()
        print('  ⚠  Weak topics (avg < 60% over last 5 attempts):')
        for w in weak:
            print(f'     • {w}')
    else:
        print()
        print('  ✅  No weak topics — great work!')
    divider()
    print()
    print('  Recent sessions (newest first):')
    print()
    if not recent:
        print('    (no sessions yet)')
    else:
        print(f'    {"Date":<12} {"Score":>6}  {"Type":<10}  Concept')
        print(f'    {"─"*10:<12} {"─"*5:>6}  {"─"*8:<10}  {"─"*20}')
        for r in reversed(recent):
            c   = store.get_content(r.content_id)
            ttl = (c.title[:28] + '…') if c and len(c.title) > 28 else (c.title if c else r.content_id[:8])
            mark = '✅' if r.correct else '❌'
            print(f'    {r.date[:10]:<12} {r.score:>5.0%}  {r.question_type:<10}  {mark} {ttl}')
    print()
    pause()


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════════

main_menu()


════════════════════════════════════════════════════════════
  🎓  AI Tutor
════════════════════════════════════════════════════════════

  [1] Manage content
  [2] Ingest new document
  [3] Study
  [4] Progress dashboard
  [0] Exit


════════════════════════════════════════════════════════════
  ⬆️   Ingest Document
════════════════════════════════════════════════════════════

  Supported: .gif, .jpeg, .jpg, .pdf, .png, .webp

  File : 10_Introduction_to_Patterns.pdf  (8156.5 KB)

  📥  Reading & ingesting "10_Introduction_to_Patterns.pdf"… this may take ~30–90s.
⬆  Uploading "10_Introduction_to_Patterns.pdf"…
🧠 ReAct — Step 1/2: Draft outline…
    [Think   1] 💭 I will start by retrieving the full topic tree to understand the existing structure and identify any relevant nodes for the new document about design patterns.
    [Act     1] ⚙️  list_tree({})
    [Observe 1] 👀 29 topic(s) in tree: [{"id": "91a7cbfd-4411-43c1-9706-0c0c774724ce", "name": "Root", "parent_id": null}, {"id": "69c6

KeyboardInterrupt: Interrupted by user